In [1]:
# dataset processing

from datasets import load_dataset

# Load the dataset
ds = load_dataset(
    "json",
    data_files={
        "train": "data/datasets/instruction_en_train.jsonl",
        "test": "data/datasets/instruction_en_eval.jsonl",
    },
)

/root/autodl-tmp/oki-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def to_conversational_pc(example):
    messages = example["messages"]
    if len(messages) < 2:
        raise ValueError(f"Expected at least 2 messages, got {len(messages)}")
    if messages[-1]["role"] != "assistant":
        raise ValueError(f"Expected final message to be assistant, got {messages[-1]['role']}")

    return {
        "prompt": messages[:-1],
        "completion": [messages[-1]],
    }


clm_ds = ds.map(to_conversational_pc, remove_columns=ds["train"].column_names)
clm_ds

Map: 100%|██████████| 200/200 [00:00<00:00, 8716.16 examples/s]


DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})

In [4]:
clm_ds["train"][0]

{'prompt': [{'role': 'user',
   'content': 'What do you call a rain storm with rain then?'}],
 'completion': [{'role': 'assistant',
   'content': "It's just a rainstorm. If it's heavy, people usually call it a downpour or a deluge. I'm not sure if there's a more specific technical term, but those should work."}]}

In [5]:
# Convert the local JSONL dataset to conversational language modeling format.
#
# Input example:
#   {"messages": [{"role": "user", ...}, {"role": "assistant", ...}], ...}
#
# Output example:
#   {"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}

def to_conversational_lm(example):
    messages = example["messages"]
    if len(messages) < 2:
        raise ValueError(f"Expected at least 2 messages, got {len(messages)}")
    if messages[0]["role"] != "user":
        raise ValueError(f"Expected first message to be user, got {messages[0]['role']}")
    if messages[-1]["role"] != "assistant":
        raise ValueError(f"Expected final message to be assistant, got {messages[-1]['role']}")

    return {
        "messages": messages,
    }


clm_ds = ds.map(to_conversational_lm, remove_columns=ds["train"].column_names)
clm_ds

Map: 100%|██████████| 200/200 [00:00<00:00, 3635.52 examples/s]


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 200
    })
})

In [7]:
clm_ds["train"][0]

{'messages': [{'role': 'user',
   'content': 'What do you call a rain storm with rain then?'},
  {'role': 'assistant',
   'content': "It's just a rainstorm. If it's heavy, people usually call it a downpour or a deluge. I'm not sure if there's a more specific technical term, but those should work."}]}